# Self-Supervised Learning — contrastive & masked pretext tasks

Companion notebook for the [Self-Supervised Learning lesson](https://ml-viz-ruby.vercel.app/courses/computer-vision/05-self-supervised-learning).

**The idea in one sentence.** Learn useful features from *unlabelled* data by
inventing a task where the labels come free from the data itself — match two
augmentations of the same image (**contrastive**), or fill in a hidden patch
(**masked reconstruction**).

**Why bother?** Labels are expensive; raw images and text are nearly free.
Self-supervised pretraining turns that free data into a representation you then
fine-tune with a *small* labelled set. It powers SimCLR/MoCo/DINO (vision),
MAE (masked images), and BERT/GPT (masked/next-token text).

**The one danger to understand: collapse.** If the pretext task can be solved by
mapping *everything* to the same vector, the model will — and learns nothing.
The two families avoid it differently:

- **Contrastive** (InfoNCE) adds **negatives**: collapse makes every pair
  equally similar, which the loss punishes.
- **Masked / predictive** avoids collapse because the target is the *actual
  hidden content*, which a constant can't reproduce.

Roadmap: (1) InfoNCE from scratch, (2) validate it against `scipy` cross-entropy,
(3) visualise collapse vs. good representations, (4) a masked-reconstruction
task, (5) tradeoffs & gotchas, (6) your turn.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

def normalize(Z):
    return Z / np.linalg.norm(Z, axis=1, keepdims=True)

## 1 — InfoNCE contrastive loss

For a batch, view A and view B are two augmentations of the same images (so row i of A matches row i
of B). The loss pulls each A toward its matching B and pushes it from all other B's — a softmax over
similarities with the diagonal as the target.

In [ ]:
def info_nce(Za, Zb, tau=0.1):
    Za, Zb = normalize(Za), normalize(Zb)
    logits = (Za @ Zb.T) / tau          # (B,B) similarity matrix
    logits -= logits.max(1, keepdims=True)
    p = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
    B = len(Za)
    return -np.mean(np.log(p[np.arange(B), np.arange(B)] + 1e-9))

B, d = 8, 16
base = rng.normal(size=(B, d))
Za = base + 0.05 * rng.normal(size=(B, d))     # two augmented views of the SAME images
Zb = base + 0.05 * rng.normal(size=(B, d))
Zrand = rng.normal(size=(B, d))                 # unrelated
print(f'loss, matched views:   {info_nce(Za, Zb):.3f}  (low: positives align)')
print(f'loss, mismatched views: {info_nce(Zrand, Zb):.3f}  (high)')

### Validate: InfoNCE is cross-entropy with the diagonal as the label

`info_nce` is a one-directional softmax cross-entropy where the correct class
for row $i$ is column $i$. We confirm it equals `scipy.special.log_softmax`
evaluated on the diagonal — same object, different spelling.

In [ ]:
from scipy.special import log_softmax as sp_log_softmax

def info_nce_scipy(Za, Zb, tau=0.1):
    Za, Zb = normalize(Za), normalize(Zb)
    logits = (Za @ Zb.T) / tau
    B = len(Za)
    return -sp_log_softmax(logits, axis=1)[np.arange(B), np.arange(B)].mean()

ours   = info_nce(Za, Zb)
theirs = info_nce_scipy(Za, Zb)
print(f'from-scratch info_nce : {ours:.6f}')
print(f'scipy cross-entropy   : {theirs:.6f}')
assert np.isclose(ours, theirs, atol=1e-6), 'InfoNCE must equal softmax cross-entropy'
print('\n✅ InfoNCE = softmax cross-entropy over similarities, label = the matching view')

## 2 — Representation collapse

A 'shortcut' encoder that maps everything to the same vector makes positives perfectly aligned — but
the *negatives* in InfoNCE punish it, because then every pair is equally similar and the softmax
can't separate the true positive. Negatives are what prevent collapse.

In [ ]:
collapsed = np.ones((B, d))                      # encoder output: identical for all inputs
print(f'InfoNCE on collapsed reps: {info_nce(collapsed, collapsed):.3f}')
print(f'InfoNCE on good reps:      {info_nce(Za, Zb):.3f}')
print(f'log(B) = {np.log(B):.3f}  <- collapsed loss hits this floor: every pair looks identical')
print('\nThe negatives make collapse a HIGH-loss solution, so the model avoids it.')

### Visualize — collapsed vs. good representations

Two similarity matrices. **Left**: a collapsed encoder (all outputs identical) —
every cell is ~1, so no positive stands out and the softmax cannot pick the
diagonal → loss stuck at the $\log B$ floor. **Right**: a good encoder — the
diagonal (matched views) dominates, which is exactly what drives the loss down.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8'})

S_collapsed = normalize(collapsed) @ normalize(collapsed).T
S_good      = normalize(Za) @ normalize(Zb).T

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, M, ttl in [(axes[0], S_collapsed, f'collapsed  (loss={info_nce(collapsed, collapsed):.2f} = log B)'),
                   (axes[1], S_good, f'good reps  (loss={info_nce(Za, Zb):.2f})')]:
    im = ax.imshow(M, cmap='viridis', vmin=-1, vmax=1)
    ax.set_title(ttl, fontsize=11); ax.set_xlabel('view B'); ax.set_ylabel('view A')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()
print('Left: uniform — no diagonal to lock onto. Right: bright diagonal = the signal InfoNCE rewards.')

## 3 — A masked-reconstruction pretext task

Masked modeling (MAE/BERT) hides part of the input and predicts it. We mask patches of a signal and
measure reconstruction error — a model that learns structure reconstructs better than the mean
baseline.

In [ ]:
def mask_patches(x, mask_ratio, seed):
    r = np.random.default_rng(seed)
    mask = r.random(len(x)) < mask_ratio
    visible = x.copy(); visible[mask] = 0.0
    return visible, mask

x = np.sin(np.linspace(0, 6*np.pi, 60))         # a structured signal
visible, mask = mask_patches(x, mask_ratio=0.5, seed=1)
# a 'model' that knows the structure interpolates the masked points; baseline predicts the mean
recon = visible.copy(); recon[mask] = np.interp(np.where(mask)[0], np.where(~mask)[0], x[~mask])
err_model = np.mean((recon[mask] - x[mask])**2)
err_base  = np.mean((x.mean() - x[mask])**2)
print(f'reconstruction MSE  (structure-aware): {err_model:.3f}')
print(f'reconstruction MSE  (mean baseline):   {err_base:.3f}')
print('Lower error means the representation captured the signal structure -> useful features.')

### Visualize — the masked-reconstruction task

The structure-aware "model" interpolates the masked points from the visible
ones and lands close to the truth; the mean baseline ignores structure and pays
for it. This is the 1-D cartoon of what MAE does with image patches.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
t = np.arange(len(x))
ax.plot(t, x, color='#6366f1', lw=2, label='ground truth', zorder=1)
ax.scatter(t[~mask], x[~mask], color='#22d3ee', s=28, label='visible', zorder=3)
ax.scatter(t[mask], recon[mask], color='#f97316', s=28, marker='x',
           label=f'reconstructed (MSE={err_model:.3f})', zorder=3)
ax.axhline(x.mean(), color='#94a3b8', ls=':', label=f'mean baseline (MSE={err_base:.3f})')
ax.set_xlabel('position'); ax.set_ylabel('signal'); ax.legend(fontsize=9,
    facecolor='#1a1d27', edgecolor='#2a2d3a', labelcolor='#e2e8f0')
ax.set_title('Masked reconstruction: predict the hidden points', fontsize=12)
plt.tight_layout(); plt.show()
print(f'structure-aware MSE {err_model:.3f}  vs  mean-baseline MSE {err_base:.3f} '
      f'({err_base/err_model:.1f}x better)')

## Tradeoffs & gotchas

| Method family | How it avoids collapse | Cost |
|---------------|------------------------|------|
| Contrastive (SimCLR/MoCo, InfoNCE) | in-batch **negatives** | needs large batches / memory bank |
| Non-contrastive (BYOL/SimSiam) | **stop-gradient** + predictor, no negatives | subtle; can still collapse if misconfigured |
| Masked (MAE/BERT) | target is the **real hidden content** | high mask ratio needed to make it non-trivial |

Two demos below: (1) temperature controls how hard the negatives push, and
(2) the mask ratio controls how hard — and how useful — the pretext task is.

### Gotcha 1 — temperature $\tau$ controls the strength of the negatives

Small $\tau$ sharpens the softmax so the *hardest* negative dominates the
gradient (strong but noisy signal); large $\tau$ flattens it toward uniform,
where even good representations score near the $\log B$ chance floor. We sweep
$\tau$ on the *same good* embeddings.

In [ ]:
print(f'{"tau":>6} | {"info_nce":>9} | {"log B":>6}')
for tau in [0.02, 0.1, 0.5, 2.0]:
    print(f'{tau:6.2f} | {info_nce(Za, Zb, tau=tau):9.3f} | {np.log(B):6.3f}')
print('\nGood reps score low at small tau (sharp contrast) and drift toward the log B')
print('floor as tau grows and the softmax can no longer tell positive from negatives.')

### Gotcha 2 — the mask ratio sets the difficulty of the pretext task

Mask too little and the task is trivial (a near-copy is visible next door);
mask too much and there is not enough context to reconstruct. MAE famously uses
a **high** ratio (~75%) so the task forces real feature learning. We sweep the
ratio and watch reconstruction error rise as the task gets harder.

In [ ]:
print(f'{"mask %":>7} | {"recon MSE":>9} | {"# masked":>8}')
for mr in [0.15, 0.35, 0.5, 0.75, 0.9]:
    vis, m = mask_patches(x, mask_ratio=mr, seed=1)
    if m.sum() == 0 or (~m).sum() < 2:
        continue
    rec = vis.copy()
    rec[m] = np.interp(np.where(m)[0], np.where(~m)[0], x[~m])
    mse = np.mean((rec[m] - x[m])**2)
    print(f'{mr*100:6.0f}% | {mse:9.3f} | {m.sum():8d}')
print('\nHigher mask ratio -> fewer visible anchors -> harder reconstruction. The sweet spot')
print('is high enough to be non-trivial but low enough to stay solvable (MAE uses ~75%).')

## ✏️ Your turn

**Exercise.** Implement `cosine_sim_matrix(Za, Zb)` (pairwise cosine similarities between two batches
of L2-normalized rows) and `contrastive_accuracy(Za, Zb)` — the fraction of rows whose *most similar*
partner in Zb is its true match (the diagonal). This is the standard retrieval-style SSL probe.

In [ ]:
def cosine_sim_matrix(Za, Zb):
    # TODO(you): normalize rows, return the (B,B) matrix of cosine similarities Za_i . Zb_j
    return ...

def contrastive_accuracy(Za, Zb):
    # TODO(you): fraction of rows i whose argmax over Zb is i (the true positive)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
S = cosine_sim_matrix(Za, Zb)
assert S.shape == (B, B)
assert np.allclose(np.diag(cosine_sim_matrix(Za, Za)), 1.0)     # self-similarity is 1
assert contrastive_accuracy(Za, Zb) == 1.0                       # matched views retrieve each other
assert contrastive_accuracy(Zrand, Zb) < 1.0                     # random views don't
print('\u2713 similarity matrix and contrastive accuracy are correct')

<details>
<summary>Solution</summary>

```python
def cosine_sim_matrix(Za, Zb):
    return normalize(Za) @ normalize(Zb).T

def contrastive_accuracy(Za, Zb):
    S = cosine_sim_matrix(Za, Zb)
    return np.mean(S.argmax(axis=1) == np.arange(len(Za)))
```

Pulling matched views together (high diagonal similarity) while pushing apart mismatches is the
whole game; the negatives are what stop the trivial collapse solution from winning.

</details>

## Key takeaways

- **Self-supervision invents free labels** from unlabelled data: match two
  augmentations (contrastive) or predict a hidden patch (masked).
- **InfoNCE is softmax cross-entropy** over a similarity matrix, with the
  matching view as the label — we verified it against `scipy` exactly.
- **Collapse is the enemy.** Contrastive methods block it with **negatives**
  (collapse hits the $\log B$ floor); masked methods block it because a constant
  can't reproduce real hidden content; BYOL/SimSiam use **stop-gradient** instead.
- **The knobs:** temperature $\tau$ (how hard negatives push) and mask ratio
  (how hard the pretext task is — MAE uses ~75%).
- The learned representation is the *product*; the pretext task is scaffolding
  you throw away, keeping the encoder for a small-labelled-data fine-tune.